<a href="https://colab.research.google.com/github/arletedelira-TI/analise-llm-rag-e-agentes-de-ia/blob/main/analise_de_documentos_com_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalando Bibliotecas


In [ ]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph


# Conectando API KEY


In [ ]:
from google.colab import userdata
import os
api_key=userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

#Importando Chat Groq do LangChain

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [ ]:
prompt = '''
	messages: Segundo a NASA quais seriam os benefícios científicos de ir para Marte?
'''

In [ ]:
resposta = llm.invoke(prompt)

In [ ]:
resposta

# Adicionando documento pdf para análise

In [ ]:
url = 'https://raw.githubusercontent.com/arletedelira-TI/analise-llm-projeto-nasa/main/document-nasa-s-moon-to-mars-strategy-and-objectives-development.pdf'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(url)
pages = []
for page in loader.lazy_load():
	pages.append(page)

In [ ]:
print(f"{pages[0].metadata}\n")

In [ ]:
print(pages[0].page_content)

# Criando uma base de dados vetorial

## Obs: Lembre-se de habilitar a opção GPU no google colab

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embed_model = HuggingFaceEmbeddings (model_name="mixedbread-ai/mxbai-embed-large-v1")

In [ ]:
vector_store = InMemoryVectorStore.from_documents(pages, embed_model)

# Gerando uma resposta com RAG

In [ ]:
docs = vector_store.similarity_search("Objective Development Process, k=2")

In [ ]:
for doc in docs:
	print(f'Page {doc.metadata["page"]}: {doc.page_content[:300]}\n')

# Pipeline de RAG

In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template = """ You're a helpful assistant that only gives answers bases on the given context. If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}

Answer:

"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from IPython.display import display, Markdown

In [ ]:
response = chain.invoke("What are the Objectives Development Process?")
display(Markdown(response))

# Adicionando mais fontes de informação

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def pega_contexto(query: str) -> str:
  """Pega o contexto baseado em uma pesquisa."""
  retriever = vector_store.as_retriever()
  resultado = retriever.invoke(query)
  return resultado

In [ ]:

def carrega_pdf(url: str):
  loader = PyPDFLoader(url)
  pages = []
  for page in loader.load():
    pages.append(page)

  vectorstore = InMemoryVectorStore.from_documents(pages, embed_model)
  return vectorstore

In [ ]:
vector_store_agriculture = carrega_pdf('https://raw.githubusercontent.com/arletedelira-TI/analise-llm-rag-e-agentes-de-ia/main/agriculture.pdf')

In [ ]:
vector_store_dengue = carrega_pdf('https://raw.githubusercontent.com/arletedelira-TI/analise-llm-rag-e-agentes-de-ia/main/dengue.pdf')

In [ ]:
@tool
def pega_contexto_agriculture(query: str) -> str:
  """Pega o contexto sobre agricultura baseado em uma pesquisa."""
  retriever = vector_store_agriculture.as_retriever()
  resultado = retriever.invoke(query)
  return resultado

In [ ]:
@tool
def pega_contexto_dengue(query: str) -> str:
  """Pega o contexto sobre dengue baseado em uma pesquisa."""
  retriever = vector_store_dengue.as_retriever()
  resultado = retriever.invoke(query)
  return resultado

In [ ]:
tools = [pega_contexto, pega_contexto_agriculture, pega_contexto_dengue]

In [ ]:
pega_contexto_dengue.invoke("Cases of dengue we had since the beginning of 2025?")


In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
system_prompt = """You're a helpful assistant that only gives answer bases on the given context. If the answer is not in the context, say "I don't know"
  - pega_contexto: Tool that returns the context based on the users query if the query is about NASA and space travels.
  - pega_contexto_agriculture: Tool that returns the context based on the users query if the query is about agriculture.
  - pega_contexto_dengue: Tool that returns the context based on the users query if the query is about dengue.
"""

In [ ]:
agente_pdf = create_react_agent(model=llm, tools=tools, prompt=system_prompt)

In [ ]:
agente_pdf.invoke({"messages": [("user", "What are the Objectives Development Process?")]})

In [ ]:
agente_pdf.invoke({"messages": [("user", "What cases dengue?")]})

# Definindo um grafo com memória

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition

from langgraph.prebuilt import ToolNode
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
grafo = StateGraph(MessagesState)

In [ ]:
grafo.add_node("assistente", agente_pdf)
grafo.add_node("tools", ToolNode(tools))

In [ ]:
grafo.add_edge(START, "assistente")
grafo.add_conditional_edges("assistente",tools_condition)

In [ ]:
grafo.add_edge("tools", "assistente")

In [ ]:
grafo.add_edge("assistente", END)

In [ ]:
memoria = MemorySaver()
app = grafo.compile(checkpointer=memoria)

In [ ]:
Image(app.get_graph().draw_mermaid_png())

In [ ]:
def chat_com_memoria(mensagem_usuario: str, thread_id="1", verbose=False):
  config = {"configurable": {"thread_id": thread_id}}
  messages = app.invoke({"messages": [HumanMessage(content=mensagem_usuario)]}, config)
  if verbose:
      for message in messages['messages']:
        message.pretty_print()
  else:
      messages['messages'][-1].pretty_print()

In [ ]:
chat_com_memoria(mensagem_usuario="Why is agriculture crucial for India's economy, and what's its current need?", thread_id="3", verbose = True)

In [ ]:
chat_com_memoria(mensagem_usuario="What is the planet NASA is going?", thread_id="3", verbose = False)